# VIBE robust-prune runs: viewer

Plots and tables for the runs of `vibe_pipeline.py` (`sbatch src/vibe_pipeline.slurm <dataset>`), which downloads a VIBE dataset, builds a Gaussian with the same covariance spectrum, builds the robust-prune graph of both, and writes everything to `GRAPH_DIR`. Nothing is computed here except plots, so this runs anywhere `GRAPH_DIR` is visible, including while jobs are still running.

- **Runs**: every run in `GRAPH_DIR` (anything with a spectrum or results file), with how far each graph has got.
- **Results**: the statistics table of each finished run.
- **Per run**: covariance spectra (dataset vs Gaussian), out-degree distributions, and coverage: the mean fraction of points still uncovered after each edge, from the `(neighbor, uncovered)` pairs on each adjacency line.

Graphs in the old format, `source [neighbor1, ...]` with no uncovered counts, have no coverage plot; `deprecated_vibe.ipynb` is the viewer for those.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from paths import RESULTS_DIR
from vibe import coverage_curve, find_runs, gaussian_spectrum, read_graph, run_info

GRAPH_DIR = RESULTS_DIR / "vibe_graphs"     # where vibe_pipeline.py wrote its runs (its --graph-dir)
SEED = 0                 # the run's --seed (only used to regenerate a Gaussian spectrum that was not saved)
PRUNE_ALPHA = 1.0        # the run's --alpha
PLOT_PARTIAL = True      # also plot graphs that are still being built (sources done so far)
COLORS = {"dataset": "C0", "gaussian": "C1"}   # matplotlib's default colors, the same in every panel

runs = [run_info(GRAPH_DIR, name, PRUNE_ALPHA) for name in find_runs(GRAPH_DIR)]
print(f"{len(runs)} run(s) in {GRAPH_DIR}: {', '.join(r['name'] for r in runs)}")

## Runs

In [ ]:
def progress(done, n):
    return f"{done:,}" + (f" / {n:,} ({100 * done / n:.0f}%)" if n else "")

pd.DataFrame([{
    "run": r["name"], "n": r["n"], "d": r["d"],
    "d_eff": r["lam"].sum() ** 2 / (r["lam"] ** 2).sum() if r["lam"] is not None else None,
    "dataset graph": progress(r["graphs"]["dataset"][1], r["n"]),
    "Gaussian graph": progress(r["graphs"]["gaussian"][1], r["n"]),
    "finished": r["results"] is not None,
} for r in runs]).style.format({"n": "{:,.0f}", "d": "{:,.0f}", "d_eff": "{:.1f}"}, na_rep="–").hide(axis="index")

## Results

Finished runs only (`<name>-results.csv`). In-degree and reciprocity cover complete graphs.

In [ ]:
finished = [r["results"].assign(run=r["name"]) for r in runs if r["results"] is not None]
if finished:
    table = pd.concat(finished, ignore_index=True).set_index(["run", "dataset"])
    display(table.style.format(
        {"n": "{:,}", "d_eff": "{:.1f}", "d_eff / d": "{:.3f}", "sources": "{:,}", "edges": "{:,}",
         "out mean": "{:.1f}", "out median": "{:.0f}", "out p99": "{:.0f}", "out max": "{:.0f}",
         "in median": "{:.0f}", "in p99": "{:.0f}", "in max": "{:.0f}", "in zero": "{:,.0f}",
         "reciprocity": "{:.3f}"}, na_rep="–"))
else:
    print("no finished runs yet")

## Per run

For each run: spectrum, out-degree distribution and coverage. Reading a graph parses its whole adjacency list (about a minute for 1M sources).

In [ ]:
for r in runs:
    name, n, d, lam = r["name"], r["n"], r["d"], r["lam"]
    lam_syn = gaussian_spectrum(GRAPH_DIR, name, lam, n, SEED)
    graphs = {}
    for label, (adj_path, done) in r["graphs"].items():
        if done and (PLOT_PARTIAL or done == n):
            graphs[label] = read_graph(adj_path, uncovered=True)

    plt.figure(figsize=(15, 4.2))
    plt.subplot(1, 3, 1)
    if lam is not None:
        k = np.arange(1, d + 1)
        plt.plot(k, lam / lam[0], color=COLORS["dataset"], label="dataset")
        if lam_syn is not None:
            plt.plot(k, lam_syn / lam_syn[0], color=COLORS["gaussian"], ls="--", label="gaussian (sample)")
        plt.xlim(1, d)
        plt.legend()
    plt.ylim(0, 1)
    plt.xlabel("rank k")
    plt.ylabel("λ_k / λ_1")
    plt.title("Covariance spectrum")

    plt.subplot(1, 3, 2)
    degrees = {label: np.fromiter((len(nb) for nb, _ in hoods.values()), dtype=np.int64)
               for label, hoods in graphs.items()}
    if degrees:
        hi = max(x.max() for x in degrees.values())
        bins = np.arange(0, hi + 2) if hi <= 200 else np.linspace(0, hi + 1, 101)
        for label, x in degrees.items():
            counts, edges = np.histogram(x, bins=bins)
            done = r["graphs"][label][1]
            note = "" if done == n else f", {done:,} sources so far"
            plt.stairs(counts / len(x), edges, color=COLORS[label], label=f"{label} (mean {x.mean():.1f}{note})")
        plt.legend()
    plt.xlabel("out-degree")
    plt.ylabel("fraction of sources")
    plt.title("Out-degree")

    plt.subplot(1, 3, 3)
    for label, hoods in graphs.items():
        curve = coverage_curve(hoods, n) if n else None
        if curve is not None:
            plt.plot(np.arange(1, len(curve) + 1), curve, color=COLORS[label], label=label)
    plt.yscale("log")
    plt.xlabel("edges added k")
    plt.ylabel("mean fraction still uncovered")
    plt.title("Coverage after k edges")
    if plt.gca().get_lines():
        plt.legend()

    size = f"n = {n:,}, " if n else ""
    d_eff = f", d_eff = {lam.sum() ** 2 / (lam ** 2).sum():.1f}" if lam is not None else ""
    plt.suptitle(f"{name}  ({size}d = {d}{d_eff})")
    plt.tight_layout()
    plt.show()